Előfeldolgozás

In [98]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers, Model
from keras.preprocessing.image import load_img, img_to_array

In [17]:
IMG_SIZE = 768
DATASET_PATH = "../data/"  # ide tedd a saját utad!

In [70]:
def load_image(size):
    images = []
    img_dir = os.path.join(DATASET_PATH, "train_v2")
    for index,fname in enumerate(os.listdir(img_dir)):
        if index > size:  
            break  
        img_path = os.path.join(img_dir, fname)
        image = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
        image = img_to_array(image) / 255.0
        images.append(image)
    return np.array(images)

In [104]:
def load_mask(size):
    df_mask = pd.read_csv(DATASET_PATH+"train_ship_segmentations_v2.csv")
    masks = [
    np.array(list(map(int, mask.split()))) 
        if not (df_mask['EncodedPixels'].isna()[index]) 
        else -1
        for index,mask in enumerate(df_mask["EncodedPixels"][:size])]
    return np.array(masks)
    

In [81]:
def load_dataset(size = 100):
    images = load_image(size)
    masks = load_mask(size)
    return images, masks


In [86]:
X, y = load_dataset(100)

In [96]:
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    return x


def encoder_block(x, filters):
    f = conv_block(x, filters)
    p = layers.MaxPooling2D((2,2))(f)
    return f, p


def decoder_block(x, skip, filters):
    x = layers.UpSampling2D((2,2))(x)
    x = layers.Concatenate()([x, skip])
    x = conv_block(x, filters)
    return x



def build_unet():
    inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3))

    # Encoder
    s1, p1 = encoder_block(inputs, 64)
    s2, p2 = encoder_block(p1, 128)
    s3, p3 = encoder_block(p2, 256)
    s4, p4 = encoder_block(p3, 512)

    # Bottleneck
    b = conv_block(p4, 1024)

    # Decoder
    d1 = decoder_block(b, s4, 512)
    d2 = decoder_block(d1, s3, 256)
    d3 = decoder_block(d2, s2, 128)
    d4 = decoder_block(d3, s1, 64)

    outputs = layers.Conv2D(1, 1, activation="sigmoid")(d4)

    return Model(inputs, outputs)


In [99]:
model = build_unet()
model.summary()

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_2 (InputLayer)        [(None, 768, 768, 3)]        0         []                            
                                                                                                  
 conv2d_19 (Conv2D)          (None, 768, 768, 64)         1792      ['input_2[0][0]']             
                                                                                                  
 conv2d_20 (Conv2D)          (None, 768, 768, 64)         36928     ['conv2d_19[0][0]']           
                                                                                                  
 max_pooling2d_4 (MaxPoolin  (None, 384, 384, 64)         0         ['conv2d_20[0][0]']           
 g2D)                                                                                       

In [102]:
model.fit(
    X, y,
    batch_size=4,
    epochs=20,
)

ValueError: Failed to find data adapter that can handle input: <class 'numpy.ndarray'>, (<class 'list'> containing values of types {"<class 'int'>", "<class 'numpy.ndarray'>"})